In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/byt5-small"  
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [2]:
from peft import LoraConfig, get_peft_model, TaskType

In [3]:
lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, 
                         r=16, #rand(8,16,32)
                        lora_alpha=32,
                        lora_dropout=0.1,
                        target_modules = ["q", "v", "k", "o"],
                        bias="none")

In [4]:
# Inject LoRA into model
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [5]:
model.print_trainable_parameters()

trainable params: 2,375,680 || all params: 302,013,440 || trainable%: 0.7866


In [6]:
#DataCollator

from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest")

In [7]:
#Trainer

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(output_dir="./byt5-lora", per_device_train_batch_size=2, per_device_eval_batch_size=2,
                                 gradient_accumulation_steps=8, learning_rate=1e-4, num_train_epochs=12, fp16=True,
                                 label_smoothing_factor = 0.1, eval_strategy="steps", save_steps=1000, eval_steps=1000, 
                                logging_steps=100, report_to="none")

In [8]:
def tokenize_fn(batch):
    model_inputs = tokenizer(batch["src"], padding="longest", truncation=True, max_length=128)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(batch["tgt"], padding="longest", truncation=True, max_length=128)

    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    model_inputs["decoder_attention_mask"] = labels["attention_mask"]

    return model_inputs

In [9]:
from datasets import load_from_disk

train_ds = load_from_disk("hf_train_ds")
test_ds  = load_from_disk("hf_test_ds")


split = train_ds.train_test_split(test_size=0.1,   # 10% validation
    seed=42
)

train_ds_new = split["train"]
val_ds   = split["test"]


tokenized_train = train_ds_new.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)

tokenized_val = val_ds.map(tokenize_fn,batched=True, remove_columns=val_ds.column_names)

In [64]:
len(train_ds)

1561

In [63]:
len(test_ds)

4

In [10]:
# model.gradient_checkpointing_enable()
# model.config.use_cache = False

In [11]:
trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=tokenized_train, 
                  eval_dataset=tokenized_val, tokenizer=tokenizer, data_collator=data_collator)

C:\Users\kumar\AppData\Local\Temp\ipykernel_10056\2552561512.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=tokenized_train,


In [12]:
trainer.train()

Step,Training Loss,Validation Loss
1000,654091468.800000,nan


TrainOutput(global_step=1056, training_loss=4299531963.575758, metrics={'train_runtime': 1820.0442, 'train_samples_per_second': 9.257, 'train_steps_per_second': 0.58, 'total_flos': 3900512769933312.0, 'train_loss': 4299531963.575758, 'epoch': 12.0})

In [13]:
model.save_pretrained("byt5-lora-adapter")
tokenizer.save_pretrained("byt5-lora-adapter")

('byt5-lora-adapter\\tokenizer_config.json',
 'byt5-lora-adapter\\special_tokens_map.json',
 'byt5-lora-adapter\\added_tokens.json')

In [14]:
# base model + adapter = final model

In [15]:
#INFERENCE

In [16]:
#Load base model again

In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

base_model_name = "google/byt5-small"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

In [18]:
#Load LoRA adapter on top

In [19]:
model = PeftModel.from_pretrained(base_model,"byt5-lora-adapter")

model.eval()

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(384, 1472)
      (encoder): T5Stack(
        (embed_tokens): Embedding(384, 1472)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=1472, out_features=384, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1472, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=384, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
             

In [20]:
text = "ahi atta"   # Akkadian example

inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [21]:
inputs

{'input_ids': tensor([[100, 107, 108,  35, 100, 119, 119, 100,   1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [48]:
import torch
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_length=60, num_beams=4,no_repeat_ngram_size=2, repetition_penalty=1.4,length_penalty=1.0,early_stopping=True, do_sample=False)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

attaches antonounetsafaithawala)forabamautuaka..()).__(_)_


In [49]:
from torch.utils.data import DataLoader
import torch

def translate_val(texts, batch_size=16):
    preds = []
    loader = DataLoader(texts, batch_size=batch_size)

    for batch in loader:
        # prompts = [f"Translate literally from Akkadian to English.\n{t}"for t in batch]

        # prompts = [f"Translate to English only:\n{t}"for t in batch]

        inputs = tokenizer(batch,return_tensors="pt",padding=True,truncation=True,max_length=96).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_length=60, num_beams=4, do_sample=False,
                repetition_penalty=1.4, no_repeat_ngram_size=2, length_penalty=1.0,early_stopping=True
            )

        decoded = tokenizer.batch_decode(outputs,skip_special_tokens=True)

        preds.extend([normalize_pred(p) for p in decoded])

    return preds

In [50]:
def normalize_pred(s):
    s = s.replace("  ", " ")
    s = s.replace("the the", "the")
    return s.strip()

In [51]:
val_srcs = val_ds["transliteration"]
val_refs = val_ds["translation"]

In [52]:
import re

def clean(p):
    p = re.sub(r"<0x..>", "", p)
    p = re.sub(r"\s+", " ", p)
    return p.strip()

preds = [clean(p) for p in translate_val(val_srcs)]

In [53]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(preds, [val_refs])
print("BLEU:", bleu.score)

BLEU: 3.062487030477116e-06


In [58]:
chrf = sacrebleu.corpus_chrf(
    preds,
    [val_refs],
)
print("chrF++:", chrf.score)

chrF++: 1.6335399423288524


In [59]:
import math

final_score = math.sqrt(bleu.score * chrf.score)
print("Final (Kaggle-style):", final_score)

Final (Kaggle-style): 0.0022366704914109376
